### Lab Assignment: Python Warmup and Logfile Analytics

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: January 23, 2026

---

This lab consists of two parts:

- Part 1 is the Python warmup
- Part 2 is the logfile analytics in PySpark

Answer the questions in this assignment, showing all code and solutions.

**Total points: 20**

---

### Part 1: Python Warmup

1) (1 PT) Rename this notebook to JupyterTutorial_[your_initials], where you will enter your initials in place of [your_initials].

2) (1 PT) In the cell below, enter a list of data science topics you find interesting.  Use the markdown style (you will need to change the style from the Code style).

3) (1 PT) In the cell below, enter the following Python list:  

some_vals = [1, 6, 10, 44]  

You will use the Code style, run the cell, and print the list.

In [172]:
some_vals = [1, 6, 10, 44]  

4) (1 PT) Use a list comprehension to return a filtered list containing only the values greater than 6.  
Call this list *some_vals_filtered* and print it.

In [173]:
some_vals_filtered = [x for x in some_vals if x > 6]
some_vals_filtered

[10, 44]

Next, a small pandas dataframe is constructed.

In [174]:
import pandas as pd

df = pd.DataFrame({'first_name': ['Andy','Crystal'],
                   'domain_facebook' : [1,1],
                   'domain_foursquare' : [0,0],
                   'age' : [20, 32]})
df

,first_name,domain_facebook,domain_foursquare,age
0,Andy,1,0,20
1,Crystal,1,0,32


5) (1 PT) In the cell below, write a list comprehension that returns the fields names in the dataframe `df` containing the string *domain*.  Run the cell to verify the correct result.

In [175]:
field_names = [x for x in df.columns if "domain" in x]
field_names

['domain_facebook', 'domain_foursquare']

6) (1 PT) Use the list comprehension from (5) to index into `df` and show the data for columns containing *domain*

In [176]:
data_domain = df[[x for x in field_names]]

data_domain

,domain_facebook,domain_foursquare
0,1,0
1,1,0


7) (1 PT) In the cell below, print the *domain_facebook* column

In [177]:
print(data_domain['domain_facebook'])

0    1
1    1
Name: domain_facebook, dtype: int64


8) (1 PT) In the cell below, print the row with index 1.

In [178]:
print(data_domain['domain_facebook'][1])

1


9) (1 PT) Next, you will cube the *age* column of `df` and assign the result to a new column called *agecube*.

Specifically, call the `apply` method with a `lambda function` inside to cube the *age* column.  
Print the dataframe.

In [179]:
df['age'] = df['age'].apply(lambda x: pow(x,3))

df

,first_name,domain_facebook,domain_foursquare,age
0,Andy,1,0,8000
1,Crystal,1,0,32768


10) (1 PT) Given the list of strings below, form one string, placing semicolons between each word.  It should look like this:  

`'the;quick;brown;fox'`

Print the resulting string.

In [180]:
some_list = ['the','quick','brown','fox']

In [181]:
# results = ''

# for i in some_list:
#     results+= i+";"

# print(results)

#OR

results = ''.join([word + ';' for word in some_list])
results

'the;quick;brown;fox;'

---

### Part 2: Logfile Analytics

Import modules for Spark Session and regex 

Note: regexes can be used to search strings for patterns. Here is a [reference](https://realpython.com/regex-python/?utm_source=chatgpt.com).

In [182]:
import os,sys
#before setting

print("Before:", os.environ.get("PYSPARK_PYTHON"))

#I added system environment variables to avoid Java error, idk
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = "C:\\Program Files\\Microsoft\\jdk-17.0.20.8-hotspot\\"

Before: c:\Users\tbrai\AppData\Local\Programs\Python\Python314\python.exe


In [183]:
from pyspark.sql import SparkSession
import re

#custom settings
spark = SparkSession.builder \
    .master("local[2]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()
# spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [184]:
print("After:", os.environ.get("PYSPARK_PYTHON"))

After: c:\Users\tbrai\AppData\Local\Programs\Python\Python314\python.exe


11) (1 PT) Read in the logfile.txt data

In [185]:
logfile = sc.textFile("logfile.txt")

In [186]:
type(logfile)

pyspark.core.rdd.RDD

12) (1 PT) Count the number of rows of data

In [187]:
logfile.count()

100000

13) (2 PTS) Show the first five lines containing WARN.  
    Write code to count and print the total number of lines containing WARN.

In [188]:
def safe_filter(line):
    try:
        return line is not None and "ERROR" in line
    except Exception:
        return False

In [189]:
#Filter logic with safeguards
warncounts = logfile.filter(lambda line: line is not None and "WARN" in line)

# Collect the filtered data to the driver
py = warncounts.collect()

for line in py[0:5]:
    print(line)

2026-01-01T08:00:09.764000 [WARN] scheduler: heartbeat ip=92.174.240.101 latency_ms=475 trace=nw6i1m40mwvc seq=19
2026-01-01T08:00:12.828000 [WARN] api: partition reassigned ip=84.161.222.95 latency_ms=4691 trace=yht26vhtm0eh seq=26
2026-01-01T08:00:14.718000 [WARN] auth: job completed ip=230.79.10.241 latency_ms=3732 trace=tedo8mtq5qu9 seq=29
2026-01-01T08:00:20.259000 [WARN] api: shuffle completed ip=212.54.74.42 latency_ms=4063 trace=ej2u544h7nwc seq=40
2026-01-01T08:00:28.523000 [WARN] auth: cache miss ip=242.94.66.209 latency_ms=753 trace=feqvh0k4dwun seq=57


In [190]:
#Total number of lines with WARN level logging
print(f"Total number of WARN level logs: {warncounts.count()}")

Total number of WARN level logs: 14823


14) (2 PTS) Write a word count program to count the number of each of these log levels:  

- WARN
- INFO
- DEBUG
- ERROR

In [191]:
# log level list
log_levels = ["WARN","INFO","DEBUG","ERROR"]
line_filter = ""

# Search through logs using each key word and count lines
for level in log_levels:
    line_filter = logfile.filter(lambda line: line is not None and level in line)
    print(f"\nTotal number of {level} level logs: {line_filter.count()}")


Total number of WARN level logs: 14823

Total number of INFO level logs: 70083

Total number of DEBUG level logs: 5143

Total number of ERROR level logs: 9951


15. (2 PTS) Return the three log lines with the highest latency. This is reported in the log as `latency_ms`.

Note: There may be more than three lines tied for highest latency, in which case, just show three records.

# Exploratory Section: Created bigram for parsing latency times

In [192]:
# create bi-gram from words 
bigrams = logfile \
            .map(lambda x: x.split()) \
            .flatMap(lambda x: [(x[i],x[i+1]) for i in range(0,len(x)-1)])

In [193]:
a = bigrams.collect()

a[0:5]

[('2026-01-01T08:00:00.888000', '[INFO]'),
 ('[INFO]', 'api:'),
 ('api:', 'connection'),
 ('connection', 'opened'),
 ('opened', 'ip=128.219.4.246')]

In [194]:
latency_list = []

for x,y in a:
    if "latency" in str(y):
        
        print(f"{x} | {y}")
        m,n = str(y).split("=",2) #split word containing latency
        print(f"Latency: {n} ms")
        latency_list.append(n)



# cast string list to ints
latency_list = list(map(int, latency_list))

ip=128.219.4.246 | latency_ms=1340
Latency: 1340 ms
ip=90.103.199.68 | latency_ms=2948
Latency: 2948 ms
ip=175.156.135.8 | latency_ms=2797
Latency: 2797 ms
ip=108.249.194.186 | latency_ms=1414
Latency: 1414 ms
ip=201.4.186.248 | latency_ms=589
Latency: 589 ms
ip=220.159.173.22 | latency_ms=4027
Latency: 4027 ms
ip=145.35.167.142 | latency_ms=1171
Latency: 1171 ms
ip=131.7.39.193 | latency_ms=1772
Latency: 1772 ms
ip=108.158.186.214 | latency_ms=2100
Latency: 2100 ms
ip=104.183.43.60 | latency_ms=3387
Latency: 3387 ms
ip=149.173.243.230 | latency_ms=870
Latency: 870 ms
ip=143.177.73.177 | latency_ms=2604
Latency: 2604 ms
ip=166.179.136.152 | latency_ms=2064
Latency: 2064 ms
ip=117.254.27.245 | latency_ms=3031
Latency: 3031 ms
ip=192.31.104.121 | latency_ms=1282
Latency: 1282 ms
ip=27.54.122.64 | latency_ms=3220
Latency: 3220 ms
ip=127.3.240.227 | latency_ms=396
Latency: 396 ms
ip=27.3.220.172 | latency_ms=90
Latency: 90 ms
ip=206.218.206.31 | latency_ms=1140
Latency: 1140 ms
ip=92.174.2

In [195]:
type(latency_list[0])
latency_list[3]
len(latency_list)

100000

In [196]:
latency_list.sort(reverse=True)

print("Top 3 latency by (ms): \n"+str(latency_list[0:3]))

Top 3 latency by (ms): 
[5000, 5000, 5000]


# Map Lines to Index numbers

In [197]:
word_lines = logfile.zipWithIndex().map(lambda x: (x[1],x[0]))
type(word_lines)

pyspark.core.rdd.PipelinedRDD

In [198]:
word_lines.take(5)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 99.0 failed 1 times, most recent failure: Lost task 0.0 in stage 99.0 (TID 189) (BVJ-61K executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:704)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:686)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1068)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunne

In [ ]:
bag_words = word_lines.collect()

bag_words[0:5]

[(0,
  '2026-01-01T08:00:00.888000 [INFO] api: connection opened ip=128.219.4.246 latency_ms=1340 trace=l5dix9t416br seq=0'),
 (1,
  '2026-01-01T08:00:00.941000 [INFO] gateway: retrying operation ip=90.103.199.68 latency_ms=2948 trace=v387dw4c6a7x seq=1'),
 (2,
  '2026-01-01T08:00:01.464000 [INFO] api: partition reassigned ip=175.156.135.8 latency_ms=2797 trace=ozyb20q7zz4g seq=2'),
 (3,
  '2026-01-01T08:00:02.015000 [ERROR] metrics: stream started ip=108.249.194.186 latency_ms=1414 trace=lijp8mbt5w5v seq=3'),
 (4,
  '2026-01-01T08:00:02.184000 [INFO] auth: request received ip=201.4.186.248 latency_ms=589 trace=vk72ght0859f seq=4')]

In [ ]:
line_result=[] #track latency_ms in list
line_index=[] #track lines in list


# Use regex to find sections mentioning latency 
for line_no,line_desc in bag_words:
    if "latency" in str(line_desc):
        lat_list = re.findall("latency_ms=(\\d+)",line_desc)
        line_result.append(int(lat_list[0]))
     

# len(line_result)
# sc.parallelize
map_latencies = list(enumerate(line_result))

top_three = sorted(map_latencies, key=lambda x: x[1], reverse=True)[:3]
print("Top 3 latency times [Line number, Latency (ms)]: \n")
print(top_three)

Top 3 latency times [Line number, Latency (ms)]: 

[(3785, 5000), (4736, 5000), (5120, 5000)]


In [ ]:
def extract_latency(pair):
    listie = []
    for a,b in pair.collect():
    # line, idx = pair
        match = re.search(r'latency_ms=(\d+)', b)
        latency = int(match.group(1)) if match else None
        print(f"Line {a}: latency_ms={latency}")
        listie = listie.append((a,latency))
    return (listie)

In [ ]:
#---------------pyspark way-----------------------*#@

line_match = word_lines.map(lambda line: (line[0],int(re.search(r'latency_ms=(\d+)', line[1]).group(1)))) \
                        .sortBy(lambda index_line: (index_line[1]),ascending=False)\
                        .collect()[0:3]

print(line_match)
print(len(line_match))


[(3785, 5000), (4736, 5000), (5120, 5000)]
3


16. (2 PTS) Compute the average latency for each service. Ignore log entries without a latency_ms.

In [ ]:
import numpy as np

line_result=[] #track latency_ms in list
line_index=[] #track lines in list

warn_match = word_lines.map(lambda line: (line[0],int(re.search(r'WARN.*latency_ms=(\d+)', line[1]).group(1)))) \
                            .filter(lambda x: x is not None)
                        # .collect()[0:3]

# len(warn_match.collect())
warn_match

# avg_keys = ['WARN','DEBUG','INFO','ERROR']

# avg_dict = dict((key, []) for key in avg_keys)

# # Use regex to find sections mentioning latency 
# for line_no,line_desc in bag_words:
#     if "WARN" in str(line_desc):
#         lat_list = re.findall("latency_ms=(\\d+)",line_desc)
#         avg_dict['WARN'].append(int(lat_list[0]))
#         continue
#     elif "DEBUG" in str(line_desc):
#         lat_list = re.findall("latency_ms=(\\d+)",line_desc)
#         avg_dict['DEBUG'].append(int(lat_list[0]))
#         continue
#     elif "INFO" in str(line_desc):
#         lat_list = re.findall("latency_ms=(\\d+)",line_desc)
#         avg_dict['INFO'].append(int(lat_list[0]))
#         continue
#     elif "ERROR" in str(line_desc):
#         lat_list = re.findall("latency_ms=(\\d+)",line_desc)
#         avg_dict['ERROR'].append(int(lat_list[0]))


# ['WARN','DEBUG','INFO','ERROR']

# print(f"Average Latency of WARN event: {round(np.mean(avg_dict['WARN']),2)} ms")
# print(f"Average Latency of DEBUG event: {round(np.mean(avg_dict['DEBUG']),2)} ms")
# print(f"Average Latency of INFO event: {round(np.mean(avg_dict['INFO']),2)} ms")
# print(f"Average Latency of ERROR event: {round(np.mean(avg_dict['ERROR']),2)} ms")
# print(len(avg_dict['DEBUG']))

# line_result.map
# line_result[0:5]
# map_latencies = list(enumerate(line_result))

# map_latencies
# top_three = sorted(map_latencies, key=lambda x: x[1], reverse=True)[:3]
# print("Top 3 latency times [Line number, Latency (ms)]: \n")
# print(top_three)


        

PythonRDD[105] at RDD at PythonRDD.scala:59

In [ ]:
print(warn_match.collect())

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "C:\Users\tbrai\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 3622, in main
    process()
    ~~~~~~~^^
  File "C:\Users\tbrai\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "C:\Users\tbrai\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "C:\Users\tbrai\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\util.py", line 159, in wrapper
    return f(*args, **kwargs)
  File "C:\Users\tbrai\AppData\Local\Temp\ipykernel_26564\3455431436.py", line 6, i